In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Whole ESI Multi-Class Feature Engineered Random Forest with Incremental Cost Matrix (`models/rf_feng_whole_cost_matrix.ipynb`)

This notebook trains a **Whole-ESI (5-Class) Random Forest Model** using Feature Engineered inputs and an **Incremental Cost Matrix**:
- **Target Output**: Predicts all 5 emergency triage levels: **ESI 1**, **ESI 2**, **ESI 3**, **ESI 4**, and **ESI 5**.
- **Feature Set (13 Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and the **10 Clinical Feature Engineering flags** defined in `TODO.md` (`is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`).
- **NA Robustness**: Performs explicit median imputation on `age` and applies `na.roughfix` in `randomForest` to prevent missing value (`na.fail.default`) runtime errors.
- **Incremental Cost Matrix Execution**:
  - Sweeps cost multipliers $C_{\text{scale}} \in \{1, 2, 3, 5, 8, 12, 20, 30, 50\}$ to heavily weight high-acuity undertriage penalties.
- **Reporting & Exports**:
  - **Per-Class Actual vs. Predicted Percentages** for every ESI class.
  - **Per-Class Accuracy, Sensitivity (Recall), and Specificity**.
  - Terminal printout & CSV export to `reports/rf_feng_whole_cost_matrix_metrics.csv`.
- **Diagnostic Plots**:
  - Cost Matrix Training/Tuning Curve Plots saved to `plots/rf_feng_whole_cost_matrix_sensitivity_curve.png`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(randomForest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Compute 13 FE Features & Clean Missing Values
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

# Clean predictor vectors and impute missing age with median
median_age <- suppressWarnings(median(raw_df$age, na.rm = TRUE))
if (is.na(median_age)) median_age <- 45
age_clean  <- ifelse(is.na(raw_df$age), median_age, raw_df$age)

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
gender_vec[is.na(gender_vec)] <- 0

cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
cc_bd_vec[is.na(cc_bd_vec)] <- 0

# Compute 10 Clinical Feature Engineering flags + Age + Gender + cc_breathingdifficulty
df_feng <- data.frame(
  age                     = age_clean,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

esi_clean <- as.character(raw_df[[target_col]])
df_feng$target_esi <- factor(esi_clean, levels = c("1", "2", "3", "4", "5"))
df_feng <- df_feng[!is.na(df_feng$target_esi), ]
df_feng <- na.omit(df_feng)  # Ensure complete dataset for Random Forest

cat(sprintf("Whole-ESI 5-Class FE Dataset Ready: %d rows x %d cols\n", nrow(df_feng), ncol(df_feng)))
cat("Feature Engineered Input Columns (13):\n", paste(setdiff(names(df_feng), "target_esi"), collapse = ", "), "\n")
cat("Whole-ESI Target Distribution:\n")
print(table(df_feng$target_esi))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning & Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

in_train_val <- createDataPartition(df_feng$target_esi, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]

rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_esi, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Incremental Cost Matrix Random Forest Training & Detailed Per-Class Evaluation
# ---------------------------------------------------------
cost_multipliers <- c(1, 2, 3, 5, 8, 12, 20, 30, 50)
esi_classes <- c("1", "2", "3", "4", "5")
all_reports_df <- data.frame()

reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
csv_report_path <- file.path(reports_dir, "rf_feng_whole_cost_matrix_metrics.csv")

cat("\n========================================================================================\n")
cat("   INCREMENTAL COST MATRIX RANDOM FOREST EVALUATION REPORT (PER-CLASS METRICS)   \n")
cat("========================================================================================\n")

for (c_scale in cost_multipliers) {
  set.seed(config$training$random_state)
  
  # Define Class Weight Penalty (Cost Matrix scaling for high acuity ESI 1 & 2)
  class_weights <- c("1" = c_scale * 2.0, "2" = c_scale * 1.2, "3" = 1.0, "4" = 1.0, "5" = 1.0)
  
  rf_fit <- randomForest(
    target_esi ~ .,
    data = train_df,
    classwt = class_weights,
    ntree = 100,
    importance = FALSE,
    na.action = na.roughfix
  )
  
  pred_val <- predict(rf_fit, newdata = val_df)
  val_actual <- val_df$target_esi
  
  N_total <- length(val_actual)
  cm <- confusionMatrix(pred_val, val_actual)
  cm_tab <- cm$table  # Rows: Predicted, Columns: Actual
  
  cat(sprintf("\n=====================================================================\n"))
  cat(sprintf("   COST MULTIPLIER (C = %.1f) - VALIDATION EVALUATION BENCHMARK\n", c_scale))
  cat(sprintf("=====================================================================\n"))
  cat(sprintf("Overall Accuracy: %.4f (%.2f%%)\n\n", cm$overall["Accuracy"], cm$overall["Accuracy"] * 100))
  
  # Build Per-Class Metrics Table
  per_class_list <- data.frame()
  for (cls in esi_classes) {
    tp <- if (cls %in% rownames(cm_tab) && cls %in% colnames(cm_tab)) cm_tab[cls, cls] else 0
    actual_cnt <- sum(cm_tab[, cls])
    pred_cnt   <- sum(cm_tab[cls, ])
    
    fn <- actual_cnt - tp
    fp <- pred_cnt - tp
    tn <- N_total - (tp + fp + fn)
    
    act_pct  <- (actual_cnt / N_total) * 100
    pred_pct <- (pred_cnt / N_total) * 100
    diff_cnt <- pred_cnt - actual_cnt
    
    cls_acc  <- (tp + tn) / N_total
    cls_sens <- if ((tp + fn) > 0) tp / (tp + fn) else 0
    cls_spec <- if ((tn + fp) > 0) tn / (tn + fp) else 0
    
    per_class_list <- rbind(per_class_list, data.frame(
      Cost_Multiplier = c_scale,
      Class           = cls,
      Actual_Count    = actual_cnt,
      Actual_Pct      = round(act_pct, 2),
      Predicted_Count = pred_cnt,
      Predicted_Pct   = round(pred_pct, 2),
      Diff_Count      = diff_cnt,
      Class_Accuracy  = round(cls_acc, 4),
      Sensitivity     = round(cls_sens, 4),
      Specificity     = round(cls_spec, 4)
    ))
  }
  
  # Terminal Printout for Current Cost Step
  print(per_class_list[, c("Class", "Actual_Count", "Actual_Pct", "Predicted_Count", "Predicted_Pct", "Diff_Count", "Class_Accuracy", "Sensitivity", "Specificity")])
  
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm_tab)
  
  all_reports_df <- rbind(all_reports_df, per_class_list)
}

# Export Full Report to CSV
write.csv(all_reports_df, file = csv_report_path, row.names = FALSE)
cat(sprintf("\n=== FULL INCREMENTAL COST MATRIX METRICS REPORT WRITTEN TO CSV: %s ===\n", csv_report_path))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Plot Cost Matrix Training / Tuning Curve
# ---------------------------------------------------------
# Plot 1: Per-Class Sensitivity vs Cost Multiplier
p_sens <- ggplot(all_reports_df, aes(x = Cost_Multiplier, y = Sensitivity, color = Class, group = Class)) +
  geom_line(size = 1.2) +
  geom_point(size = 3) +
  theme_minimal() +
  scale_color_brewer(palette = "Set1") +
  labs(title = "Whole-ESI Cost Matrix Tuning: Per-Class Sensitivity vs. Cost Multiplier",
       subtitle = "Shows how increasing high-acuity cost penalties boosts ESI 1 & ESI 2 Sensitivity",
       x = "Cost Matrix Multiplier (C)", y = "Sensitivity (Recall)") +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "right")

plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)

ggsave(file.path(plots_dir, "rf_feng_whole_cost_matrix_sensitivity_curve.png"), plot = p_sens, width = 9, height = 5, dpi = 300)
cat("Sensitivity Curve Plot saved to: plots/rf_feng_whole_cost_matrix_sensitivity_curve.png\n")

print(p_sens)

# Plot 2: Per-Class Specificity vs Cost Multiplier
p_spec <- ggplot(all_reports_df, aes(x = Cost_Multiplier, y = Specificity, color = Class, group = Class)) +
  geom_line(size = 1.2) +
  geom_point(size = 3) +
  theme_minimal() +
  scale_color_brewer(palette = "Set1") +
  labs(title = "Whole-ESI Cost Matrix Tuning: Per-Class Specificity vs. Cost Multiplier",
       subtitle = "Specificity performance across ESI classes as cost penalty scales",
       x = "Cost Matrix Multiplier (C)", y = "Specificity") +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "right")

ggsave(file.path(plots_dir, "rf_feng_whole_cost_matrix_specificity_curve.png"), plot = p_spec, width = 9, height = 5, dpi = 300)
cat("Specificity Curve Plot saved to: plots/rf_feng_whole_cost_matrix_specificity_curve.png\n")

print(p_spec)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Final Random Forest Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

# Train final model with balanced cost multiplier (e.g. C = 5)
opt_cost <- 5.0
class_weights_opt <- c("1" = opt_cost * 2.0, "2" = opt_cost * 1.2, "3" = 1.0, "4" = 1.0, "5" = 1.0)

set.seed(config$training$random_state)
rf_final <- randomForest(
  target_esi ~ .,
  data = train_df,
  classwt = class_weights_opt,
  ntree = 200,
  importance = TRUE,
  na.action = na.roughfix
)

model_path <- file.path(deploy_dir, "rf_feng_whole_cost_matrix_model.rds")
saveRDS(list(model = rf_final, preproc = preproc, cost_multiplier = opt_cost), file = model_path)
cat("Final Whole-ESI Cost Matrix Random Forest model saved to:", model_path, "\n")